**# Question 3:**

LSTM for Autoregressive Text Generation

https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/05_autoregressive/01_lstm/lstm.ipynb

## **1.Importing Libaries**


In [1]:
import numpy as np  # load numerical array library
import json  # load JSON handling tools
import re  # load pattern matching tools
import string  # load text utilities for punctuation

import tensorflow as tf  # load TensorFlow framework

from tensorflow.keras import layers  # import neural network layers
from tensorflow.keras import models  # import model structure tools
from tensorflow.keras import callbacks  # import training control tools
from tensorflow.keras import losses  # import loss functions

In [2]:
VOCAB_SIZE = 10000  # limit vocabulary to top 10,000 words
MAX_LEN = 200  # set maximum sequence length to 200 tokens
EMBEDDING_DIM = 100  # set embedding vector size to 100
N_UNITS = 128  # set LSTM hidden units to 128
VALIDATION_SPLIT = 0.2  # use 20 percent data for validation
SEED = 42  # fix random seed for reproducibility
LOAD_MODEL = False  # do not load pre-trained model
BATCH_SIZE = 32  # set training batch size to 32
EPOCHS = 25  # train model for 25 epochs

These values set model size, training length, and data split for LSTM training.

In [3]:
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_aee4a19abe6d69b91a6e9a811d3a4c8c"

In [4]:

# download dataset
!kaggle datasets download -d hugodarwood/epirecipes

# unzip dataset
!unzip -q epirecipes.zip

# check files
!ls

Dataset URL: https://www.kaggle.com/datasets/hugodarwood/epirecipes
License(s): unknown
epirecipes.zip: Skipping, found more recently modified local copy (use --force to force download)
replace epi_r.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace full_format_recipes.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace recipe.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace utils.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
epi_r.csv	full_format_recipes.json  sample_data
epirecipes.zip	recipe.py		  utils.py


In [5]:
import json

# load dataset
with open("./full_format_recipes.json", "r") as json_data:
    recipe_data = json.load(json_data)

# filter and build text corpus
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x
    and x["title"] is not None
    and "directions" in x
    and x["directions"] is not None
]

In [7]:
# count total recipes in filtered dataset
n_recipes = len(filtered_data)

# print number of loaded recipes
print(f"{n_recipes} recipes loaded")

20111 recipes loaded


In [8]:
example = filtered_data[9]
print(example)

Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas  | Chop enough parsley leaves to measure 1 tablespoon; reserve. Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan, covered, 5 minutes. Meanwhile, sprinkle gelatin over water in a medium bowl and let soften 1 minute. Strain broth through a fine-mesh sieve into bowl with gelatin and stir to dissolve. Season with salt and pepper. Set bowl in an ice bath and cool to room temperature, stirring. Toss ham with reserved parsley and divide among jars. Pour gelatin on top and chill until set, at least 1 hour. Whisk together mayonnaise, mustard, vinegar, 1/4 teaspoon salt, and 1/4 teaspoon pepper in a large bowl. Stir in celery, cornichons, and potatoes. Pulse peas with marjoram, oil, 1/2 teaspoon pepper, and 1/4 teaspoon salt in a food processor to a coarse mash. Layer peas, then potato salad, over ham.


In [9]:
# split punctuation into separate tokens
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)  # add spaces around punctuation
    s = re.sub(" +", " ", s)  # remove extra spaces
    return s

# apply preprocessing to all recipes
text_data = [pad_punctuation(x) for x in filtered_data]

# select one sample recipe
example_data = text_data[9]
example_data

'Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas | Chop enough parsley leaves to measure 1 tablespoon ; reserve . Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan , covered , 5 minutes . Meanwhile , sprinkle gelatin over water in a medium bowl and let soften 1 minute . Strain broth through a fine - mesh sieve into bowl with gelatin and stir to dissolve . Season with salt and pepper . Set bowl in an ice bath and cool to room temperature , stirring . Toss ham with reserved parsley and divide among jars . Pour gelatin on top and chill until set , at least 1 hour . Whisk together mayonnaise , mustard , vinegar , 1 / 4 teaspoon salt , and 1 / 4 teaspoon pepper in a large bowl . Stir in celery , cornichons , and potatoes . Pulse peas with marjoram , oil , 1 / 2 teaspoon pepper , and 1 / 4 teaspoon salt in a food processor to a coarse mash . Layer peas , then potato salad , over ham . '

In [10]:
# convert text list into TensorFlow dataset
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)  # create dataset from text list
    .batch(BATCH_SIZE)  # group samples into batches
    .shuffle(1000)  # shuffle data for better training order
)

In [11]:
# create text vectorization layer for token indexing
vectorize_layer = layers.TextVectorization(
    standardize="lower",  # convert text to lowercase
    max_tokens=VOCAB_SIZE,  # limit vocabulary size
    output_mode="int",  # convert tokens into integer IDs
    output_sequence_length=MAX_LEN + 1,  # fix sequence length
)

In [12]:
# build vocabulary from training dataset
vectorize_layer.adapt(text_ds)

# extract learned vocabulary list
vocab = vectorize_layer.get_vocabulary()

In [13]:
# print token to word mapping for first 10 vocabulary items
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")

0: 
1: [UNK]
2: .
3: ,
4: and
5: to
6: in
7: the
8: with
9: a


In [14]:
# convert sample text into integer token sequence
example_tokenised = vectorize_layer(example_data)

# display numeric representation of text
print(example_tokenised.numpy())

[  26   16  557    1    8  298  335  189    4 1054  494   27  332  228
  235  262    5  594   11  133   22  311    2  332   45  262    4  671
    4   70    8  171    4   81    6    9   65   80    3  121    3   59
   12    2  299    3   88  650   20   39    6    9   29   21    4   67
  529   11  164    2  320  171  102    9  374   13  643  306   25   21
    8  650    4   42    5  931    2   63    8   24    4   33    2  114
   21    6  178  181 1245    4   60    5  140  112    3   48    2  117
  557    8  285  235    4  200  292  980    2  107  650   28   72    4
  108   10  114    3   57  204   11  172    2   73  110  482    3  298
    3  190    3   11   23   32  142   24    3    4   11   23   32  142
   33    6    9   30   21    2   42    6  353    3 3224    3    4  150
    2  437  494    8 1281    3   37    3   11   23   15  142   33    3
    4   11   23   32  142   24    6    9  291  188    5    9  412  572
    2  230  494    3   46  335  189    3   20  557    2    0    0    0
    0 

In [15]:
# convert each text into input and target sequences for training
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)  # add extra dimension for batch processing
    tokenized_sentences = vectorize_layer(text)  # convert text to integer tokens
    x = tokenized_sentences[:, :-1]  # input sequence (all words except last)
    y = tokenized_sentences[:, 1:]  # target sequence (all words shifted by 1)
    return x, y

# apply transformation to dataset
train_ds = text_ds.map(prepare_inputs)

In [16]:
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 100)      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 128)      │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,407,248 (9.18 MB)

 Trainable params: 2,407,248 (9.18 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
if LOAD_MODEL:
    # model.load_weights('./models/model')
    lstm = models.load_model("./models/lstm", compile=False)

In [18]:
loss_fn = losses.SparseCategoricalCrossentropy()
lstm.compile("adam", loss_fn)

In [19]:
# callback class for generating text during training
class TextGenerator(callbacks.Callback):

    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word  # map index to word

        self.word_to_index = {
            word: index for index, word in enumerate(index_to_word)
        }  # map word to index

    def sample_from(self, probs, temperature):
        probs = probs ** (1 / temperature)  # control randomness
        probs = probs / np.sum(probs)  # normalize probabilities
        return np.random.choice(len(probs), p=probs), probs  # sample token

    def generate(self, start_prompt, max_tokens, temperature):
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]  # convert prompt to tokens

        sample_token = None
        info = []

        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])  # model input sequence
            y = self.model.predict(x, verbose=0)  # predict next token probabilities

            sample_token, probs = self.sample_from(y[0][-1], temperature)  # pick next token

            start_tokens.append(sample_token)  # append token to sequence

            start_prompt = start_prompt + " " + self.index_to_word[sample_token]  # build text

        print(f"\ngenerated text:\n{start_prompt}\n")  # show output

        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=1.0)  # generate sample text each epoch

In [21]:
# create model checkpoint to save weights after each epoch
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",  # correct Keras format
    save_weights_only=True,  # save only weights
    save_freq="epoch",  # save after each epoch
    verbose=0
)

In [22]:
# setup TensorBoard logging directory for training metrics
tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# initialize text generator using vocabulary for decoding predictions
text_generator = TextGenerator(vocab)

In [23]:
lstm.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)

Epoch 1/25
629/629 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 5.0178
generated text:
recipe for lemongrass pie | preheat water in the half cup put 4 to each open ) if inch , cover sugar . diameter encircle water until pie maple clean and drizzled with a oil / chocolate 6 slices . transfer the together until octopus minutes crosswise to a thick alternately the third pan in it joints portion day . cool ) wedges and top , chopped medium tablespoon 

629/629 ━━━━━━━━━━━━━━━━━━━━ 39s 56ms/step - loss: 4.1670
Epoch 2/25
629/629 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 3.1161
generated text:
recipe for pork and tuscan whole | whisk together sugar , and lime juice , salt and heat . the medium and florets to medium and thyme to boil and blend . mix vanilla and toss occasionally . generously 1 day ahead . wash bread with it starts to 1 month . before candy panko ) or sprinkle with buttermilk and heat chicken on 1 puff baking to rimmed baking bowl . add oil and fry dressing with heavy cherries , 

In [26]:
import os

# create directory for saving model
os.makedirs("./models", exist_ok=True)

# save full keras model
lstm.save("./models/lstm.keras")

In [39]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, j in zip(p_sorted, i_sorted):
            print(f"{vocab[j]}:\t{np.round(100*p,2)}%")
        print("--------\n")

info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /",
    max_tokens=10,
    temperature=1.0
)
print_probs(info, vocab)

info = text_generator.generate(
    "recipe for roasted vegetables | chop 1 /",
    max_tokens=10,
    temperature=0.2
)
print_probs(info, vocab)

info = text_generator.generate(
    "recipe for chocolate ice cream |",
    max_tokens=7,
    temperature=1.0
)
print_probs(info, vocab)

info = text_generator.generate(
    "recipe for chocolate ice cream |",
    max_tokens=7,
    temperature=0.2
)
print_probs(info, vocab)


generated text:
recipe for roasted vegetables | chop 1 / 2 cup


generated text:
recipe for roasted vegetables | chop 1 / 2 cup


generated text:
recipe for chocolate ice cream | put


generated text:
recipe for chocolate ice cream | in



**CONCLUSION**

The model repeats similar recipe steps because training data assigns high probability to common cooking tokens and low temperature makes it choose the highest probability token each step, so outputs stay similar across runs.
